In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

class UnderwaterDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform

        self.images = sorted(os.listdir(image_dir))
        self.masks = sorted(os.listdir(mask_dir))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.images[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        image = Image.open(image_path).convert("RGB")
        mask = Image.open(mask_path)


        if self.transform:
            image = self.transform(image)


        mask = mask.resize((256, 256), resample=Image.NEAREST)

        mask = torch.tensor(np.array(mask), dtype=torch.long)
        mask = remap_mask(mask)

        return image, mask


image_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

base_dir = "/kaggle/input/q3-stage3-2026"


data_dir = base_dir
while "dataset" in os.listdir(data_dir):
    data_dir = os.path.join(data_dir, "dataset")

image_dir = os.path.join(data_dir, "images")
mask_dir = os.path.join(data_dir, "masks")

print("Image dir:", image_dir)
print("Mask dir:", mask_dir)

full_dataset = UnderwaterDataset(
    image_dir=image_dir,
    mask_dir=mask_dir,
    transform=image_transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)


images, masks = next(iter(train_loader))

plt.figure(figsize=(10, 4))
for i in range(3):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(2, 3, i + 4)
    plt.imshow(masks[i], cmap="jet")
    plt.title("Mask")
    plt.axis("off")

plt.show()

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
# TO DO
import torch
import segmentation_models_pytorch as smp

num_classes = 8
model = smp.Unet(encoder_name="efficientnet-b1", encoder_weights="imagenet", in_channels=3, classes=num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
# TO DO
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = running_loss / len(dataloader)
    return epoch_loss

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item()
    epoch_loss = running_loss / len(dataloader)
    return epoch_loss

In [ ]:
# TO DO
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 5

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss = validate(
        model, val_loader, criterion, device
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f}")



plt.figure()
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training and Validation Loss")
plt.show()


In [ ]:
# TO DO
model.eval()
images, masks = next(iter(val_loader))
images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)
    preds = torch.argmax(outputs, dim=1)

images = images.cpu()
masks = masks.cpu()
preds = preds.cpu()

num_samples = 3

plt.figure(figsize=(12, 4 * num_samples))

for i in range(num_samples):

    plt.subplot(num_samples, 3, i * 3 + 1)
    plt.imshow(images[i].permute(1, 2, 0))
    plt.title("Image")
    plt.axis("off")


    plt.subplot(num_samples, 3, i * 3 + 2)
    plt.imshow(masks[i], cmap="jet")
    plt.title("Ground Truth")
    plt.axis("off")


    plt.subplot(num_samples, 3, i * 3 + 3)
    plt.imshow(preds[i], cmap="jet")
    plt.title("Prediction")
    plt.axis("off")

plt.show()